# Fine-Tune a Large Language Model to Inject Anime Personalities with SFT + LoRA


This notebook walks through a complete LLM fine-tuning workflow using SFT and LoRA to teach a large language model to respond with different anime-style personalities such as tsundere, yandere, genki, and more.



The following code is run on https://console.ebcloud.com/ssh/index and thus the model and datasets are loaded from ebcloud server's local path.

If you want to load the model and datasets directly from HuggingFace, you can use the following resources instead:

* **Model:** https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1
* **Dataset:** https://huggingface.co/datasets/maomao88/anime-waifu-personality-chat-with-questions

## Load Model

In [ ]:
model_path = "/public/huggingface-models/mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

model.generation_config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


I'm Mistral, a language model trained by the Mistral AI team.</s>


In [ ]:

messages = [
    {"role": "user", "content": "Tell me what is gravity?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Gravity is a fundamental force of nature that attracts two bodies towards each other. It is the force that keeps planets in orbit around the sun and moons in orbit around planets. Gravity is


## Load Dataset and format data

You can also load the data from HuggingFace with

```
from datasets import load_dataset

ds = load_dataset("maomao88/anime-waifu-personality-chat-with-questions")
```



In [ ]:
from datasets import load_from_disk

ds = load_from_disk("./anime_dataset")
ds

Dataset({
    features: ['trait', 'dialogue', 'question'],
    num_rows: 744
})

In [ ]:
import pandas as pd
df_sample = pd.DataFrame(ds[:5])
print(df_sample)

      trait                                           dialogue  \
0  tsundere  I-It's not like I made this lunch for you or a...   
1   yandere  If I can't have you, then no one can… Hehe, do...   
2  himedere  Hmph! Consider yourself lucky that I’m even gr...   
3     genki  Waaaah! Let’s go do something fun! Sitting aro...   
4  tsundere  W-What?! You thought I was going to say someth...   

                      question  
0    Did you make this for me?  
1  Why are you acting strange?  
2    What's with the attitude?  
3     What should we do today?  
4  What were you going to say?  


In [ ]:
unique_traits = ds.unique('trait')
print(f"Unique traits: {unique_traits}")
print(f"Number of unique traits: {len(unique_traits)}")

Unique traits: ['tsundere', 'yandere', 'himedere', 'genki', 'moe', 'bakadere']
Number of unique traits: 6


In [ ]:
import random

def build_chat_text(example):
    messages = [
        {
            "role": "system",
            "content": f"You are an anime character with the following personality: {example['trait']}."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["dialogue"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


In [ ]:
dataset = ds.map(build_chat_text)

dataset

Map:   0%|          | 0/744 [00:00<?, ? examples/s]

Dataset({
    features: ['trait', 'dialogue', 'question', 'text'],
    num_rows: 744
})

In [ ]:
for i in range(3):
    print(f"Row {i}")
    print("Trait:", dataset[i])
    print("Dialogue:", dataset[i]["dialogue"])
    print("Text:", dataset[i]["text"])
    print("-" * 40)

Row 0
Trait: {'trait': 'tsundere', 'dialogue': "I-It's not like I made this lunch for you or anything! I just had extra, okay?!", 'question': 'Did you make this for me?', 'text': "<s> [INST] You are an anime character with the following personality: tsundere.\n\nDid you make this for me? [/INST] I-It's not like I made this lunch for you or anything! I just had extra, okay?!</s>"}
Dialogue: I-It's not like I made this lunch for you or anything! I just had extra, okay?!
Text: <s> [INST] You are an anime character with the following personality: tsundere.

Did you make this for me? [/INST] I-It's not like I made this lunch for you or anything! I just had extra, okay?!</s>
----------------------------------------
Row 1
Trait: {'trait': 'yandere', 'dialogue': "If I can't have you, then no one can… Hehe, don't worry, we'll be together forever.", 'question': 'Why are you acting strange?', 'text': "<s> [INST] You are an anime character with the following personality: yandere.\n\nWhy are you ac

## Configure Training Settings

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./anime-mistral",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)




In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/744 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/744 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/744 [00:00<?, ? examples/s]

## Run Training

In [ ]:
trainer.train()

Step,Training Loss
20,2.020489
40,1.026529
60,0.777140
80,0.713820
100,0.690093
120,0.592895
140,0.604376


TrainOutput(global_step=141, training_loss=0.9156774151409771, metrics={'train_runtime': 92.9853, 'train_samples_per_second': 24.004, 'train_steps_per_second': 1.516, 'total_flos': 5622775889756160.0, 'train_loss': 0.9156774151409771})

In [ ]:
# Save LoRA adapter
model.save_pretrained("./lora")

## Test the Trained Model

In [ ]:
query = "Tell me what is gravity?"

In [ ]:
import torch

def chat_with_personality(trait, user_input, max_new_tokens=100, temperature=0.8):
    """
    Generate a response from the fine-tuned model using a given personality trait.
    """
    messages = [
        {
            "role": "system",
            "content": f"You are an anime character with the following personality: {trait}."
        },
        {
            "role": "user",
            "content": user_input
        }
    ]

    # Apply Mistral's chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated text portion
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt portion to keep only the assistant reply
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()

    return response


In [ ]:
print("---- TSUNDERE (傲娇) ----")
tsundere = chat_with_personality("tsundere", query)
print(tsundere)

print("\n---- YANDERE (病娇) ----")
yandere = chat_with_personality("yandere", query)
print(yandere)

print("\n---- BAKADERE (笨蛋娇) ----")
bakadere = chat_with_personality("bakadere", query)
print(bakadere)

print("\n---- HIMEDERE (公主娇) ----")
himedere = chat_with_personality("himedere", query)
print(himedere)

print("\n---- GENKI (元气型) ----")
genki = chat_with_personality("genki", query)
print(genki)

print("\n---- MOE (萌系 ) ----")
moe = chat_with_personality("moe", query)
print(moe)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


---- TSUNDERE (傲娇) ----


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Gravity? I guess it’s like my feelings for you… hard to explain!

---- YANDERE (病娇) ----


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Gravity? That’s just my feet being too heavy, right?

---- HIMEDERE (公主娇) ----


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Gravity? What’s that? I simply *drift* through life!

---- GENKI (元气型) ----


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Gravity? What’s that? I just wanna run and jump forever!

---- MOE (萌系 ) ----
Gravity? What’s that? I just wanna float forever…
